In [73]:
import pandas as pd
import numpy as np

from sklearn.preprocessing import LabelEncoder, StandardScaler
from sklearn.model_selection import train_test_split


In [74]:
df = pd.read_csv("group_26_train.csv")

print("Shape of dataset:", df.shape)
df.head()


Shape of dataset: (100000, 35)


,f1,f2,f3,f4,f5,f6,f7,f8,f9,f10,f11,f12,f13,f14,f15,f16,f17,f18,f19,f20,f21,f22,f23,f24,f25,f26,f27,f28,f29,f30,f31,f32,f33,f34,y
0,Source2,2018-03-30 17:32:46,40.810566,-73.256042,NaN,NaN,0.000,Commerce Dr,Hauppauge,Suffolk,NY,11788-3968,KISP,52.0,NaN,29.77,10.0,NNW,13.8,0.01,Overcast,False,False,False,False,False,False,False,True,False,Day,Day,Day,Day,2
1,Source1,2022-07-27 13:42:00,45.358867,-122.762271,45.341362,-122.768971,1.252,I-5 S,Tualatin,Washington,OR,97062,KUAO,90.0,90.0,29.78,10.0,NNE,3.0,0.00,Fair,False,False,False,False,False,False,False,False,False,Day,Day,Day,Day,2
2,Source1,2022-06-06 13:00:30,41.233610,-95.954493,41.242050,-95.953603,0.585,US-75,Omaha,Douglas,NE,68105,KOMA,76.0,76.0,28.67,10.0,VAR,3.0,0.00,Mostly Cloudy,False,False,False,False,False,False,False,False,False,Day,Day,Day,Day,2
3,Source1,2021-09-07 21:50:00,34.039947,-117.505855,34.037156,-117.505860,0.193,Ranchero Dr,Fontana,San Bernardino,CA,92337,KONT,77.0,77.0,28.89,10.0,WSW,5.0,0.00,Fair,False,False,False,False,False,False,False,False,False,Night,Night,Night,Night,2
4,Source2,2019-05-23 20:52:35,47.230240,-122.448868,NaN,NaN,0.000,I-5 N,Tacoma,Pierce,WA,98418,KTCM,63.0,63.0,29.53,10.0,W,8.0,0.00,Cloudy,False,False,False,False,False,False,False,False,False,Night,Day,Day,Day,3


In [75]:
missing_values = df.isnull().sum()
missing_percent = (df.isnull().sum() / len(df)) * 100

missing_table = pd.DataFrame({
    "Missing Count": missing_values,
    "Missing Percent": missing_percent
})

missing_table = missing_table[missing_table["Missing Count"] > 0]

missing_table = missing_table.sort_values("Missing Percent", ascending=False)

missing_table


,Missing Count,Missing Percent
f5,44256,44.256
f6,44256,44.256
f20,28631,28.631
f15,25982,25.982
f19,7375,7.375
f18,2227,2.227
f17,2207,2.207
f21,2182,2.182
f14,2074,2.074
f16,1788,1.788


In [76]:
original_rows = len(df)

df.drop_duplicates(inplace=True)

print("Removed rows:", original_rows - len(df))
print("New shape:", df.shape)


Removed rows: 70
New shape: (99930, 35)


In [77]:
row_threshold = 0.2

df = df.dropna(thresh=int((1 - row_threshold) * df.shape[1]))

print("Shape after removing highly-missing rows:", df.shape)


Shape after removing highly-missing rows: (98303, 35)


In [78]:
low_nan_cols = [col for col in df.columns if 0 < df[col].isnull().mean() < 0.05]

df.dropna(subset=low_nan_cols, inplace=True)

print("Shape after removing rows with small NaN columns:", df.shape)


Shape after removing rows with small NaN columns: (96209, 35)


In [79]:
numeric_cols = df.select_dtypes(include=[np.number]).columns.tolist()
categorical_cols = df.select_dtypes(include=["object"]).columns.tolist()
bool_cols = df.select_dtypes(include=["bool"]).columns.tolist()

print("Numeric columns:", len(numeric_cols))
print(numeric_cols)

print("\nCategorical columns:", len(categorical_cols))
print(categorical_cols)

print("\nBoolean columns:", len(bool_cols))
print(bool_cols)


Numeric columns: 12
['f3', 'f4', 'f5', 'f6', 'f7', 'f14', 'f15', 'f16', 'f17', 'f19', 'f20', 'y']

Categorical columns: 14
['f1', 'f2', 'f8', 'f9', 'f10', 'f11', 'f12', 'f13', 'f18', 'f21', 'f31', 'f32', 'f33', 'f34']

Boolean columns: 9
['f22', 'f23', 'f24', 'f25', 'f26', 'f27', 'f28', 'f29', 'f30']


/tmp/ipykernel_27152/1539792205.py:2: Pandas4Warning: For backward compatibility, 'str' dtypes are included by select_dtypes when 'object' dtype is specified. This behavior is deprecated and will be removed in a future version. Explicitly pass 'str' to `include` to select them, or to `exclude` to remove them and silence this warning.
See https://pandas.pydata.org/docs/user_guide/migration-3-strings.html#string-migration-select-dtypes for details on how to write code that works with pandas 2 and 3.
  categorical_cols = df.select_dtypes(include=["object"]).columns.tolist()


In [80]:
missing_values = df.isnull().sum()
missing_percent = (df.isnull().sum() / len(df)) * 100

missing_table = pd.DataFrame({
    "Missing Count": missing_values,
    "Missing Percent": missing_percent
})

missing_table = missing_table[missing_table["Missing Count"] > 0]

missing_table = missing_table.sort_values("Missing Percent", ascending=False)

missing_table


,Missing Count,Missing Percent
f5,43032,44.727624
f6,43032,44.727624
f20,26583,27.630471
f15,23103,24.013346
f19,4903,5.096197


In [81]:
corr_3_5 = df['f3'].corr(df['f5'])
corr_4_6 = df['f4'].corr(df['f6'])
corr_14_15 = df['f14'].corr(df['f15'])

print(f"Correlation between f3 and f5: {corr_3_5:.4f}")
print(f"Correlation between f4 and f6: {corr_4_6:.4f}")
print(f"Correlation between f14 and f15: {corr_14_15:.4f}")

df.drop(columns=['f5', 'f6', 'f15'], inplace=True)

print("\nColumns 'f5', 'f6', and 'f15' have been dropped.")
print("Current columns in dataframe:", df.columns.tolist())


Correlation between f3 and f5: 1.0000
Correlation between f4 and f6: 1.0000
Correlation between f14 and f15: 0.9938

Columns 'f5', 'f6', and 'f15' have been dropped.
Current columns in dataframe: ['f1', 'f2', 'f3', 'f4', 'f7', 'f8', 'f9', 'f10', 'f11', 'f12', 'f13', 'f14', 'f16', 'f17', 'f18', 'f19', 'f20', 'f21', 'f22', 'f23', 'f24', 'f25', 'f26', 'f27', 'f28', 'f29', 'f30', 'f31', 'f32', 'f33', 'f34', 'y']


In [82]:
missing_values = df.isnull().sum()
missing_percent = (df.isnull().sum() / len(df)) * 100

missing_table = pd.DataFrame({
    "Missing Count": missing_values,
    "Missing Percent": missing_percent
})

missing_table = missing_table[missing_table["Missing Count"] > 0]

missing_table = missing_table.sort_values("Missing Percent", ascending=False)

missing_table


,Missing Count,Missing Percent
f20,26583,27.630471
f19,4903,5.096197


In [83]:
cols = ['f19', 'f20']

print("Shape:", df[cols].shape)

print("\nMissing values:")
print(df[cols].isna().sum())

print("\nMissing percentage:")
print(df[cols].isna().mean() * 100)

print("\nDescriptive statistics:")
print(df[cols].describe())

print("\nCorrelation:")
print(df[cols].corr())


Shape: (96209, 2)

Missing values:
f19     4903
f20    26583
dtype: int64

Missing percentage:
f19     5.096197
f20    27.630471
dtype: float64

Descriptive statistics:
                f19           f20
count  91306.000000  69626.000000
mean       7.681475      0.007846
std        5.881957      0.087900
min        0.000000      0.000000
25%        4.600000      0.000000
50%        7.000000      0.000000
75%       10.400000      0.000000
max      812.000000      9.990000

Correlation:
          f19       f20
f19  1.000000  0.032545
f20  0.032545  1.000000


In [84]:
from sklearn.feature_selection import mutual_info_classif

X = df[['f19','f20']]
y = df['y']

X = X.fillna(X.median())

mi = mutual_info_classif(X, y)

for col, score in zip(X.columns, mi):
    print(f"{col}: {score}")



f19: 0.027660499207051492
f20: 0.0


In [85]:
df['f19'] = df['f19'].fillna(df['f19'].median())

df.drop(columns=['f20'], inplace=True)

print("f19 missing values filled with median and f20 dropped.")
print("Current columns:", df.columns.tolist())


f19 missing values filled with median and f20 dropped.
Current columns: ['f1', 'f2', 'f3', 'f4', 'f7', 'f8', 'f9', 'f10', 'f11', 'f12', 'f13', 'f14', 'f16', 'f17', 'f18', 'f19', 'f21', 'f22', 'f23', 'f24', 'f25', 'f26', 'f27', 'f28', 'f29', 'f30', 'f31', 'f32', 'f33', 'f34', 'y']


In [86]:
if 'f2' in df.columns:
    
    df['f2'] = pd.to_datetime(df['f2'], errors='coerce')

    df['hour'] = df['f2'].dt.hour
    df['dayofweek'] = df['f2'].dt.dayofweek
    df['month'] = df['f2'].dt.month
    df['day'] = df['f2'].dt.day

    df.drop('f2', axis=1, inplace=True)

print("Date features created.")


Date features created.


In [87]:
numeric_cols = df.select_dtypes(include=[np.number]).columns.tolist()
categorical_cols = df.select_dtypes(include=["object"]).columns.tolist()
bool_cols = df.select_dtypes(include=["bool"]).columns.tolist()

print("Numeric columns:", len(numeric_cols))
print(numeric_cols)

print("\nCategorical columns:", len(categorical_cols))
print(categorical_cols)

print("\nBoolean columns:", len(bool_cols))
print(bool_cols)


Numeric columns: 12
['f3', 'f4', 'f7', 'f14', 'f16', 'f17', 'f19', 'y', 'hour', 'dayofweek', 'month', 'day']

Categorical columns: 13
['f1', 'f8', 'f9', 'f10', 'f11', 'f12', 'f13', 'f18', 'f21', 'f31', 'f32', 'f33', 'f34']

Boolean columns: 9
['f22', 'f23', 'f24', 'f25', 'f26', 'f27', 'f28', 'f29', 'f30']


/tmp/ipykernel_27152/1539792205.py:2: Pandas4Warning: For backward compatibility, 'str' dtypes are included by select_dtypes when 'object' dtype is specified. This behavior is deprecated and will be removed in a future version. Explicitly pass 'str' to `include` to select them, or to `exclude` to remove them and silence this warning.
See https://pandas.pydata.org/docs/user_guide/migration-3-strings.html#string-migration-select-dtypes for details on how to write code that works with pandas 2 and 3.
  categorical_cols = df.select_dtypes(include=["object"]).columns.tolist()


In [88]:
cat_cols = ['f1', 'f8', 'f9', 'f10', 'f11', 'f12', 'f13', 'f18', 'f21', 'f31', 'f32', 'f33', 'f34']

summary = []

for col in cat_cols:
    summary.append({
        "column": col,
        "n_unique": df[col].nunique(),
        "top_category": df[col].mode().iloc[0] if not df[col].mode().empty else None,
        "top_freq": df[col].value_counts().iloc[0] if not df[col].value_counts().empty else None
    })

summary_df = pd.DataFrame(summary)

print("===== CATEGORICAL SUMMARY =====")
print(summary_df.sort_values("n_unique", ascending=False))

print("\n===== SAMPLE DISTRIBUTION =====")
for col in cat_cols:
    print(f"\n{col}")
    print(df[col].value_counts().head(10))

print("\n===== RELATIONSHIP WITH TARGET (y) =====")
for col in cat_cols:
    print(f"\n{col}")
    print(pd.crosstab(df[col], df['y'], normalize='index').head())


===== CATEGORICAL SUMMARY =====
   column  n_unique top_category  top_freq
5     f12     35794        91761       139
1      f8     31495       I-95 N      1008
2      f9      6139        Miami      2285
6     f13      1563         KCQT      1496
3     f10      1254  Los Angeles      6531
8     f21        79         Fair     32366
4     f11        49           CA     21757
7     f18        24         CALM     12203
0      f1         3      Source1     53177
9     f31         2          Day     66586
10    f32         2          Day     71142
11    f33         2          Day     76084
12    f34         2          Day     79783

===== SAMPLE DISTRIBUTION =====

f1
f1
Source1    53177
Source2    41797
Source3     1235
Name: count, dtype: int64

f8
f8
I-95 N     1008
I-95 S      890
I-5 N       856
I-10 W      699
I-5 S       667
I-10 E      653
I-80 E      474
I-80 W      467
I-405 N     422
I-75 N      363
Name: count, dtype: int64

f9
f9
Miami          2285
Houston        2186
Los Angel

In [89]:
high_card = ['f12','f8','f9','f13','f10']
low_card = ['f21','f11','f18','f1','f31','f32','f33','f34']

for col in high_card:
    freq = df[col].value_counts() / len(df)
    df[col] = df[col].map(freq)

df = pd.get_dummies(df, columns=low_card, drop_first=True)

print("Encoding complete")
print("New shape:", df.shape)


Encoding complete
New shape: (96209, 181)


In [90]:
for col in high_card:
    if col in df.columns:
        freq = df[col].value_counts() / len(df)
        df[col] = df[col].map(freq)

existing_low_card = [c for c in low_card if c in df.columns]

df = pd.get_dummies(df, columns=existing_low_card, drop_first=True)

bool_cols = df.select_dtypes(include='bool').columns
df[bool_cols] = df[bool_cols].astype(int)

print("Encoding complete")
print("New shape:", df.shape)


Encoding complete
New shape: (96209, 181)


In [91]:
print(df.shape)
df.head()


(96209, 181)


,f3,f4,f7,f8,f9,f10,f12,f13,f14,f16,f17,f19,f22,f23,f24,f25,f26,f27,f28,f29,f30,y,hour,dayofweek,month,day,f21_Blowing Dust / Windy,f21_Blowing Snow,f21_Blowing Snow / Windy,f21_Clear,f21_Cloudy,f21_Cloudy / Windy,f21_Drizzle,f21_Drizzle and Fog,f21_Fair,f21_Fair / Windy,f21_Fog,f21_Fog / Windy,f21_Freezing Rain,f21_Haze,f21_Haze / Windy,f21_Heavy Drizzle,f21_Heavy Rain,f21_Heavy Rain / Windy,f21_Heavy Snow,f21_Heavy Snow / Windy,f21_Heavy T-Storm,f21_Heavy T-Storm / Windy,f21_Heavy Thunderstorms and Rain,f21_Heavy Thunderstorms and Snow,f21_Ice Pellets,f21_Light Drizzle,f21_Light Drizzle / Windy,f21_Light Fog,f21_Light Freezing Drizzle,f21_Light Freezing Fog,f21_Light Freezing Rain,f21_Light Ice Pellets,f21_Light Rain,f21_Light Rain / Windy,f21_Light Rain Shower,f21_Light Rain Showers,f21_Light Rain with Thunder,f21_Light Sleet,f21_Light Snow,f21_Light Snow / Windy,f21_Light Snow Shower,f21_Light Snow Showers,f21_Light Snow and Sleet,f21_Light Thunderstorms and Rain,f21_Mist,f21_Mostly Cloudy,f21_Mostly Cloudy / Windy,f21_N/A Precipitation,f21_Overcast,f21_Partly Cloudy,f21_Partly Cloudy / Windy,f21_Patches of Fog,f21_Patches of Fog / Windy,f21_Rain,f21_Rain / Windy,f21_Rain Showers,f21_Rain and Sleet,f21_Scattered Clouds,f21_Shallow Fog,f21_Showers in the Vicinity,f21_Sleet,f21_Sleet / Windy,f21_Smoke,f21_Smoke / Windy,f21_Snow,f21_Snow / Windy,f21_Snow and Sleet,f21_Snow and Sleet / Windy,f21_Squalls / Windy,f21_T-Storm,f21_T-Storm / Windy,f21_Thunder,f21_Thunder / Windy,f21_Thunder in the Vicinity,f21_Thunderstorm,f21_Thunderstorms and Rain,f21_Widespread Dust,f21_Wintry Mix,f11_AR,f11_AZ,f11_CA,f11_CO,f11_CT,f11_DC,f11_DE,f11_FL,f11_GA,f11_IA,f11_ID,f11_IL,f11_IN,f11_KS,f11_KY,f11_LA,f11_MA,f11_MD,f11_ME,f11_MI,f11_MN,f11_MO,f11_MS,f11_MT,f11_NC,f11_ND,f11_NE,f11_NH,f11_NJ,f11_NM,f11_NV,f11_NY,f11_OH,f11_OK,f11_OR,f11_PA,f11_RI,f11_SC,f11_SD,f11_TN,f11_TX,f11_UT,f11_VA,f11_VT,f11_WA,f11_WI,f11_WV,f11_WY,f18_Calm,f18_E,f18_ENE,f18_ESE,f18_East,f18_N,f18_NE,f18_NNE,f18_NNW,f18_NW,f18_North,f18_S,f18_SE,f18_SSE,f18_SSW,f18_SW,f18_South,f18_VAR,f18_Variable,f18_W,f18_WNW,f18_WSW,f18_West,f1_Source2,f1_Source3,f31_Night,f32_Night,f33_Night,f34_Night
0,40.810566,-73.256042,0.000,0.026879,0.011496,0.020663,0.285140,0.004365,52.0,29.77,10.0,13.8,0,0,0,0,0,0,0,1,0,2,17.0,4.0,3.0,30.0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,1,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,1,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,1,0,0,0,0,0,0,0,0,0,0,0,0,0,0,1,0,0,0,0,0
1,45.358867,-122.762271,1.252,0.006933,0.005893,0.005083,0.007577,0.003014,90.0,29.78,10.0,3.0,0,0,0,0,0,0,0,0,0,2,13.0,2.0,7.0,27.0,0,0,0,0,0,0,0,0,1,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,1,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,1,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0
2,41.233610,-95.954493,0.585,0.219636,0.002754,0.005062,0.008835,0.007276,76.0,28.67,10.0,3.0,0,0,0,0,0,0,0,0,0,2,13.0,0.0,6.0,6.0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,1,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,1,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,1,0,0,0,0,0,0,0,0,0,0,0
3,34.039947,-117.505855,0.193,0.219636,0.002640,0.014074,0.018086,0.004792,77.0,28.89,10.0,5.0,0,0,0,0,0,0,0,0,0,2,21.0,1.0,9.0,7.0,0,0,0,0,0,0,0,0,1,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,1,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,1,0,0,0,1,1,1,1
4,47.230240,-122.448868,0.000,0.008897,0.00

In [92]:
outlier_counts = {}
lower_bounds = {}
upper_bounds = {}

if 'numeric_cols' in locals() and numeric_cols:
    cols_to_analyze = [col for col in numeric_cols if col != 'y']
    
    print(f"Analyzing outliers for {len(cols_to_analyze)} numeric columns (excluding 'y').")

    for col in cols_to_analyze:
        if col in df.columns and pd.api.types.is_numeric_dtype(df[col]):
            Q1 = df[col].quantile(0.25)
            Q3 = df[col].quantile(0.75)
            IQR = Q3 - Q1
            
            lower = Q1 - 1.5 * IQR
            upper = Q3 + 1.5 * IQR
            
            lower_bounds[col] = lower
            upper_bounds[col] = upper
            
            outliers = ((df[col] < lower) | (df[col] > upper)).sum()
            outlier_counts[col] = outliers

    outlier_df = pd.DataFrame.from_dict(outlier_counts, orient="index", columns=["Outlier Count"])
    outlier_df["Lower Bound (1.5*IQR)"] = lower_bounds.values()
    outlier_df["Upper Bound (1.5*IQR)"] = upper_bounds.values()

    outlier_df = outlier_df[outlier_df["Outlier Count"] > 0].sort_values("Outlier Count", ascending=False)

    print("\n--- Outlier Analysis Results ---")
    if not outlier_df.empty:
        print("Outliers detected in numeric columns (sorted by count):")
        display(outlier_df)
    else:
        print("No outliers detected in numeric columns (excluding 'y') based on the 1.5*IQR rule.")

else:
    print("Error: 'numeric_cols' list is not defined or is empty. Cannot perform outlier analysis.")
    outlier_df = pd.DataFrame()

print("\n--- End of Outlier Analysis ---")


Analyzing outliers for 11 numeric columns (excluding 'y').

--- Outlier Analysis Results ---
Outliers detected in numeric columns (sorted by count):


,Outlier Count,Lower Bound (1.5*IQR),Upper Bound (1.5*IQR)
f17,18900,10.0000,10.0000
f7,12112,-0.6825,1.1375
f16,5608,28.3800,31.0200
f19,2917,-3.5000,18.1000
f14,602,8.5000,116.5000



--- End of Outlier Analysis ---


In [93]:
df_capped = df.copy()

lower_bounds = outlier_df["Lower Bound (1.5*IQR)"].to_dict()
upper_bounds = outlier_df["Upper Bound (1.5*IQR)"].to_dict()

for col, count in outlier_df["Outlier Count"].items():
    if col in df_capped.columns:
        lower = lower_bounds[col]
        upper = upper_bounds[col]
        
        df_capped[col] = np.where(df_capped[col] < lower, lower, df_capped[col])
        df_capped[col] = np.where(df_capped[col] > upper, upper, df_capped[col])
    else:
        print(f"Warning: Column '{col}' from outlier_df not found in the dataframe.")
            
df = df_capped


In [94]:
X = df.drop("y", axis=1)
y = df["y"]

print("Features shape:", X.shape)
print("Target shape:", y.shape)


Features shape: (96209, 180)
Target shape: (96209,)


In [95]:
print("Target column:", 'y')
print("\nTotal rows:", len(df))

print("\nClass counts:")
print(df['y'].value_counts().sort_index())

print("\nClass percentages:")
print((df['y'].value_counts(normalize=True).sort_index() * 100).round(3))

print("\nNumber of classes:", df['y'].nunique())

print("\nMissing values in y:")
print(df['y'].isna().sum())

summary = pd.DataFrame({
    "count": df['y'].value_counts().sort_index(),
    "percent": (df['y'].value_counts(normalize=True).sort_index()*100).round(3)
})

print("\nSummary table:")
print(summary)

print("\nSuggested test sizes to consider:")
print("0.2 (standard)")
print("0.25 (safer for evaluation)")
print("0.3 (if dataset very large)")


Target column: y

Total rows: 96209

Class counts:
y
1      822
2    76579
3    16327
4     2481
Name: count, dtype: int64

Class percentages:
y
1     0.854
2    79.597
3    16.970
4     2.579
Name: proportion, dtype: float64

Number of classes: 4

Missing values in y:
0

Summary table:
   count  percent
y                
1    822    0.854
2  76579   79.597
3  16327   16.970
4   2481    2.579

Suggested test sizes to consider:
0.2 (standard)
0.25 (safer for evaluation)
0.3 (if dataset very large)


In [96]:
from sklearn.model_selection import train_test_split

X = df.drop(columns='y')
y = df['y']

X_train, X_test, y_train, y_test = train_test_split(
    X,
    y,
    test_size=0.2,
    random_state=42,
    stratify=y
)

print("Train shape:", X_train.shape)
print("Test shape:", X_test.shape)

print("\nTrain class distribution (%):")
print((y_train.value_counts(normalize=True).sort_index()*100).round(3))

print("\nTest class distribution (%):")
print((y_test.value_counts(normalize=True).sort_index()*100).round(3))


Train shape: (76967, 180)
Test shape: (19242, 180)

Train class distribution (%):
y
1     0.855
2    79.596
3    16.970
4     2.579
Name: proportion, dtype: float64

Test class distribution (%):
y
1     0.852
2    79.597
3    16.973
4     2.578
Name: proportion, dtype: float64


In [97]:
scaler = StandardScaler()

X_train_scaled = scaler.fit_transform(X_train)
X_test_scaled = scaler.transform(X_test)

X_train_final = pd.DataFrame(X_train_scaled, columns=X.columns)
X_test_final = pd.DataFrame(X_test_scaled, columns=X.columns)

print("Data ready for modeling")
X_train_final.head()


Data ready for modeling


,f3,f4,f7,f8,f9,f10,f12,f13,f14,f16,f17,f19,f22,f23,f24,f25,f26,f27,f28,f29,f30,hour,dayofweek,month,day,f21_Blowing Dust / Windy,f21_Blowing Snow,f21_Blowing Snow / Windy,f21_Clear,f21_Cloudy,f21_Cloudy / Windy,f21_Drizzle,f21_Drizzle and Fog,f21_Fair,f21_Fair / Windy,f21_Fog,f21_Fog / Windy,f21_Freezing Rain,f21_Haze,f21_Haze / Windy,f21_Heavy Drizzle,f21_Heavy Rain,f21_Heavy Rain / Windy,f21_Heavy Snow,f21_Heavy Snow / Windy,f21_Heavy T-Storm,f21_Heavy T-Storm / Windy,f21_Heavy Thunderstorms and Rain,f21_Heavy Thunderstorms and Snow,f21_Ice Pellets,f21_Light Drizzle,f21_Light Drizzle / Windy,f21_Light Fog,f21_Light Freezing Drizzle,f21_Light Freezing Fog,f21_Light Freezing Rain,f21_Light Ice Pellets,f21_Light Rain,f21_Light Rain / Windy,f21_Light Rain Shower,f21_Light Rain Showers,f21_Light Rain with Thunder,f21_Light Sleet,f21_Light Snow,f21_Light Snow / Windy,f21_Light Snow Shower,f21_Light Snow Showers,f21_Light Snow and Sleet,f21_Light Thunderstorms and Rain,f21_Mist,f21_Mostly Cloudy,f21_Mostly Cloudy / Windy,f21_N/A Precipitation,f21_Overcast,f21_Partly Cloudy,f21_Partly Cloudy / Windy,f21_Patches of Fog,f21_Patches of Fog / Windy,f21_Rain,f21_Rain / Windy,f21_Rain Showers,f21_Rain and Sleet,f21_Scattered Clouds,f21_Shallow Fog,f21_Showers in the Vicinity,f21_Sleet,f21_Sleet / Windy,f21_Smoke,f21_Smoke / Windy,f21_Snow,f21_Snow / Windy,f21_Snow and Sleet,f21_Snow and Sleet / Windy,f21_Squalls / Windy,f21_T-Storm,f21_T-Storm / Windy,f21_Thunder,f21_Thunder / Windy,f21_Thunder in the Vicinity,f21_Thunderstorm,f21_Thunderstorms and Rain,f21_Widespread Dust,f21_Wintry Mix,f11_AR,f11_AZ,f11_CA,f11_CO,f11_CT,f11_DC,f11_DE,f11_FL,f11_GA,f11_IA,f11_ID,f11_IL,f11_IN,f11_KS,f11_KY,f11_LA,f11_MA,f11_MD,f11_ME,f11_MI,f11_MN,f11_MO,f11_MS,f11_MT,f11_NC,f11_ND,f11_NE,f11_NH,f11_NJ,f11_NM,f11_NV,f11_NY,f11_OH,f11_OK,f11_OR,f11_PA,f11_RI,f11_SC,f11_SD,f11_TN,f11_TX,f11_UT,f11_VA,f11_VT,f11_WA,f11_WI,f11_WV,f11_WY,f18_Calm,f18_E,f18_ENE,f18_ESE,f18_East,f18_N,f18_NE,f18_NNE,f18_NNW,f18_NW,f18_North,f18_S,f18_SE,f18_SSE,f18_SSW,f18_SW,f18_South,f18_VAR,f18_Variable,f18_W,f18_WNW,f18_WSW,f18_West,f1_Source2,f1_Source3,f31_Night,f32_Night,f33_Night,f34_Night
0,0.124430,-1.444342,-0.547432,1.788577,-1.021818,-0.532641,1.575203,-0.463833,-0.943458,-0.313043,0.0,0.104822,-0.112029,-0.022225,-0.28004,-0.094342,-0.005098,-0.174167,-0.032658,-0.418519,0.0,NaN,NaN,NaN,NaN,-0.006243,-0.010814,-0.010196,-0.348817,-0.345929,-0.044921,-0.023643,-0.006243,-0.71293,-0.065919,-0.116407,-0.005098,-0.005098,-0.101183,-0.015714,-0.00883,-0.062971,-0.017661,-0.024187,-0.010196,-0.036428,-0.012997,-0.017661,-0.003605,-0.006243,-0.059001,-0.00883,-0.003605,-0.01652,-0.00806,-0.020073,-0.00806,4.477330,-0.032658,-0.005098,-0.003605,-0.040333,-0.006243,-0.131842,-0.030386,0.0,-0.003605,0.0,-0.026497,-0.020394,-0.393218,-0.049219,-0.019747,-0.234475,-0.319350,-0.036606,-0.026741,-0.003605,-0.103388,-0.021931,-0.005098,0.0,-0.168951,-0.023643,-0.012997,0.0,0.0,-0.043445,-0.003605,-0.044337,-0.009537,-0.00806,-0.003605,-0.003605,-0.04621,-0.00883,-0.045498,-0.009537,-0.046631,-0.023086,-0.019415,-0.005098,-0.038003,-0.054388,-0.151503,1.850516,-0.111791,-0.096902,-0.050398,-0.041291,-0.358998,-0.148195,-0.059553,-0.036961,-0.145703,-0.09157,-0.049877,-0.066414,-0.14355,-0.091425,-0.122669,-0.01652,-0.147461,-0.160972,-0.099864,-0.042841,-0.061392,-0.214052,-0.016909,-0.062554,-0.033833,-0.13306,-0.035154,-0.055104,-0.217121,-0.125529,-0.107668,-0.152621,-0.201112,-0.048281,-0.226345,-0.00806,-0.150242,-0.290637,-0.107296,-0.198369,-0.011956,-0.125369,-0.067879,-0.045641,-0.019077,-0.226883,-0.197589,-0.184962,-0.193326,-0.117832,-0.206814,-0.188907,-0.187798,-0.213986,-0.227137,-0.137933,-0.243227,-0.198581,-0.220576,4.300603,-0.226567,-0.15875,-0.18451,-0.120855,-0.231188,-0.230126,-0.223251,-0.14888,-0.875971,-0.115023,-0.668555,-0.595623,-0.51685,-0.455995
1,1.237792,0.664381,-0.683312,-0.725600,0.101760,-0.503841,-0.600675,0.187821,-1.901195,-0.768464,0.0,-1